# 🥇 Gold — Feature Engineering

Saímos da silver com:
- `contracts.parquet` — uma linha por contrato (10k)
- `payments.parquet` — uma linha por parcela (100k)

Queremos chegar a uma tabela **`contract_features`** com uma linha por contrato e features prontas para os modelos. A pergunta é: **o que a gente sabe sobre o comportamento de pagamento que ajuda a prever sucesso/insucesso da cobrança?**

Vamos descobrir isso aos poucos, olhando os pagamentos e perguntando "o que aqui é informativo?".

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA   = Path.cwd().parents[0] / 'data'
SILVER = DATA / 'silver'
GOLD   = DATA / 'gold'
GOLD.mkdir(parents=True, exist_ok=True)

contracts = pd.read_parquet(SILVER / 'contracts.parquet')
payments  = pd.read_parquet(SILVER / 'payments.parquet')

print('contracts:', contracts.shape, '| payments:', payments.shape)

contracts: (10000, 8) | payments: (100000, 11)


## 1. Volume de parcelas por contrato

Primeira coisa óbvia: nem todo contrato tem o mesmo número de parcelas. Quantas vai de cada lado?

In [2]:
payments.groupby('id_contrato').size().describe()

count    10000.000000
mean        10.000000
std          3.161044
min          2.000000
25%          8.000000
50%         10.000000
75%         12.000000
max         23.000000
dtype: float64

Entre 2 e 23 parcelas por contrato, mediana 10. Isso já é uma feature: contratos com mais parcelas têm mais oportunidades de inadimplência.

## 2. Agregações básicas

Vamos consolidar de uma vez: total de parcelas, quantas foram pagas, total recebido, primeira e última data de pagamento.

In [3]:
agg = payments.groupby('id_contrato').agg(
    parcelas_total     = ('id_pagamento',    'count'),
    parcelas_pagas     = ('contemplado',     'sum'),
    total_pago         = ('valor_pago',      'sum'),
    primeiro_pagamento = ('data_pagamento',  'min'),
    ultimo_pagamento   = ('data_pagamento',  'max'),
).reset_index()
agg.head()

,id_contrato,parcelas_total,parcelas_pagas,total_pago,primeiro_pagamento,ultimo_pagamento
0,CONTR_2026_00001,12,2,4960.51,2025-02-17,2026-04-01
1,CONTR_2026_00002,15,6,7045.13,2025-01-24,2026-04-10
2,CONTR_2026_00003,8,4,4608.85,2025-02-16,2026-06-07
3,CONTR_2026_00004,14,4,4690.89,2025-02-20,2025-10-08
4,CONTR_2026_00005,9,1,5199.48,2025-02-28,2026-03-16


Imediatamente conseguimos derivar a **taxa de adimplência** — % das parcelas que o devedor honrou:

In [4]:
agg['taxa_adimplencia'] = agg['parcelas_pagas'] / agg['parcelas_total']
agg['taxa_adimplencia'].describe()

count    10000.000000
mean         0.299837
std          0.154140
min          0.000000
25%          0.200000
50%          0.285714
75%          0.400000
max          1.000000
Name: taxa_adimplencia, dtype: float64

## 3. Comportamento de atraso

A média de dias de atraso só faz sentido nas parcelas efetivamente pagas (em parcela não paga, `dias_atraso_pagamento` é NaN):

In [5]:
paid = payments[payments['contemplado'] == True]

atraso = (
    paid.groupby('id_contrato')['dias_atraso_pagamento']
        .mean()
        .rename('media_dias_atraso')
        .reset_index()
)
agg = agg.merge(atraso, on='id_contrato', how='left')
agg['media_dias_atraso'].describe()

count    8068.000000
mean       12.691431
std        22.113106
min        -5.000000
25%        -2.000000
50%         3.000000
75%        20.500000
max       119.000000
Name: media_dias_atraso, dtype: float64

## 4. Forma de pagamento predominante

Quando o devedor paga, ele usa que método com mais frequência? Boleto, Pix, Débito Automático? Isso pode estar correlacionado com perfil de risco.

In [6]:
metodo = (
    paid.groupby('id_contrato')['forma_pagamento']
        .agg(lambda s: s.mode().iloc[0] if len(s) else None)
        .rename('metodo_predominante')
        .reset_index()
)
agg = agg.merge(metodo, on='id_contrato', how='left')
agg['metodo_predominante'].value_counts(dropna=False)

metodo_predominante
Boleto               6327
Pix                  2212
Débito Automático     965
NaN                   496
Name: count, dtype: int64

## 5. Velocidade de pagamento

Quanto o devedor vem pagando por dia desde o primeiro pagamento? Mede a tração da recuperação:

In [7]:
ref = pd.Timestamp.now()
dias_desde_primeiro = (ref - agg['primeiro_pagamento']).dt.days.clip(lower=1)
agg['velocidade_pagamento'] = agg['total_pago'] / dias_desde_primeiro
agg['velocidade_pagamento'].describe()

count    9995.000000
mean       12.499587
std         4.821331
min         0.873786
25%         9.036067
50%        12.072184
75%        15.449280
max        37.319512
Name: velocidade_pagamento, dtype: float64

## 6. Buckets de negócio

Modelos lidam bem com numérico contínuo, mas a categorização ajuda em dashboards e explicações para humanos. Vamos derivar duas:

In [8]:
contracts['faixa_valor'] = pd.cut(
    contracts['valor_inadimplente'],
    bins=[0, 5_000, 20_000, 100_000, float('inf')],
    labels=['baixo', 'medio', 'alto', 'muito_alto'],
).astype(str)

contracts['dias_atraso_bucket'] = pd.cut(
    contracts['dias_atraso'].fillna(0),
    bins=[-1, 30, 90, 180, 365, float('inf')],
    labels=['<30d', '30-90d', '90-180d', '180-365d', '>365d'],
).astype(str)

contracts[['faixa_valor', 'dias_atraso_bucket']].apply(pd.value_counts) if False else None
contracts[['faixa_valor', 'dias_atraso_bucket']].head()

,faixa_valor,dias_atraso_bucket
0,alto,30-90d
1,medio,30-90d
2,baixo,90-180d
3,baixo,30-90d
4,baixo,30-90d


## 7. Junção contratos + agregações + label

Hora de unir tudo. O **label** para o classificador de sucesso vem do status:

- `Acordo Firmado` → `True` (sucesso)
- `Insucesso` → `False`
- `Em Aberto` e `Ajuizado` → `NaN` (não rotulado — ainda em andamento)

In [9]:
base_cols = [
    'id_contrato', 'score_risco', 'dias_atraso', 'valor_inadimplente',
    'faixa_valor', 'dias_atraso_bucket', 'regiao', 'nome_assessoria',
    'status_cobranca',
]

features = (
    contracts[base_cols]
        .merge(agg, on='id_contrato', how='left')
        .rename(columns={'dias_atraso': 'dias_atraso_inicial'})
)

features['label_sucesso'] = features['status_cobranca'].map(
    {'Acordo Firmado': True, 'Insucesso': False}
)

features['label_sucesso'].value_counts(dropna=False)

label_sucesso
NaN      5044
True     3407
False    1549
Name: count, dtype: int64

~3.400 sucessos + ~1.500 insucessos = ~4.900 amostras rotuladas. É o que vamos usar para treinar o classificador.

## 8. Features payment-level (para Modelo B)

O modelo B prevê se uma **parcela específica** será paga. Precisamos para cada parcela uma feature que diga *"como o devedor vinha se comportando até esta parcela?"* — taxa de adimplência histórica.

Atenção: tem que ser **estritamente anterior** para não vazar o futuro.

In [10]:
p = payments.sort_values(['id_contrato', 'numero_parcela']).copy()

p['paid_flag']  = p['contemplado'].fillna(False).astype(int)
p['cum_paid']   = p.groupby('id_contrato')['paid_flag'].cumsum() - p['paid_flag']
p['cum_anter']  = p.groupby('id_contrato').cumcount()  # nº de parcelas anteriores
p['taxa_adimplencia_historica'] = (p['cum_paid'] / p['cum_anter']).fillna(0)
p = p.drop(columns=['paid_flag', 'cum_paid', 'cum_anter'])

p[['id_contrato', 'numero_parcela', 'contemplado', 'taxa_adimplencia_historica']].head(15)

,id_contrato,numero_parcela,contemplado,taxa_adimplencia_historica
15191,CONTR_2026_00001,2,False,0.000000
89848,CONTR_2026_00001,11,False,0.000000
63887,CONTR_2026_00001,14,False,0.000000
68858,CONTR_2026_00001,19,False,0.000000
24058,CONTR_2026_00001,26,False,0.000000
40081,CONTR_2026_00001,27,False,0.000000
56543,CONTR_2026_00001,30,True,0.000000
83407,CONTR_2026_00001,32,False,0.142857
60047,CONTR_2026_00001,36,True,0.125000
44543,CONTR_2026_00001,38,False,0.222222


Repare: na primeira parcela `taxa_adimplencia_historica = 0` (sem histórico). Na segunda já reflete o que aconteceu na primeira. Sem vazamento.

Enriquecendo com contexto do contrato:

In [11]:
payments_features = p.merge(
    contracts[['id_contrato', 'score_risco', 'faixa_valor', 'regiao', 'nome_assessoria']],
    on='id_contrato', how='left',
)

payments_features['dias_since_envio'] = (
    payments_features['data_vencimento']
    - payments_features.groupby('id_contrato')['data_vencimento'].transform('min')
).dt.days

payments_features.shape

(100000, 17)

## 9. Persistência gold

In [12]:
features.to_parquet(GOLD / 'contract_features.parquet', index=False)
payments_features.to_parquet(GOLD / 'payments_features.parquet', index=False)

for f in sorted(GOLD.glob('*.parquet')):
    print(f'  {f.name:30s} {f.stat().st_size/1024**2:6.2f} MB')

  contract_features.parquet        0.40 MB
  payments_features.parquet        1.60 MB


Gold pronto. As próximas etapas:

- `04_supabase_load.ipynb` — carregar as três camadas em Postgres
- `05_eda_analytics.ipynb` — explorar visualmente o que temos
- Notebooks `02_ml/*` — treinar os modelos com as features deste arquivo